# E48 --- a regua: em que moeda a lei andou

**A tentativa.** O capitulo mede o andar da familia ajustada na moeda do PARAMETRO (o orcamento da
variacao). Mas a promessa mais velha da casa --- o corte --- e calibrada na LEI da janela, e
"quanto a lei andou" muda de numero com a regua.

**O que se mede.**

1. o andar da lei janela a janela, nos tres mercados, em tres reguas: variacao total, divergencia
   de Kullback-Leibler e Wasserstein-1;
2. o chao: o mesmo andar em mundos que nunca mudam --- quanto o sorteio mostra sozinho;
3. as duas amarras que amarram as reguas (Pinsker e o diametro);
4. o brinquedo declarado da cauda, e a consequencia: o corte puro perde a assinatura no mundo que
   anda, o corte com margem a devolve pagando largura.

**Convencoes** (AGENTS.md §7 e §9): um experimento por caderno, parametros no topo marcados com
"brinque com", algoritmo em frevolab, resultado em lab/resultados/E48_regua.json, figuras em .pdf
e .png.

In [1]:
# <- brinque com: SERIES, JANELA, CELULAS, C, RECENTES, CAUDA, MUNDOS_PARADOS, MUNDOS_ANDANDO, N_MUNDO, FATOR_MUNDO, SORTES_BRINQUEDO, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, lei, mudanca, promessa, volatilidade

SERIES = ("sp500.csv", "ibov.csv", "btc.csv")   # os tres mercados do capitulo
JANELA = 252                   # o corte: um ano de pregoes
CELULAS = 20                   # as celulas partilhadas da variacao total
C = 1.0                        # a constante declarada que traduz andar em largura
RECENTES = 63                  # os dias recentes que medem o andar
CAUDA = 0.05                   # a promessa: um dia em vinte
MUNDOS_PARADOS = 100           # o chao das tres reguas
MUNDOS_ANDANDO = 100           # a cobertura do corte puro e do corte com margem
N_MUNDO = 12000                # o tamanho de cada mundo
FATOR_MUNDO = 2.0              # o quanto ela anda
SORTES_BRINQUEDO = 200         # as sortes do brinquedo da cauda
SEMENTE = 115                  # E40..E47 usam 107..114

SIGMA = mudanca.SIGMA_PADRAO
print("frevolab %s | mundos de %d dias, %d parados e %d andando | semente %d"
      % (frevolab.VERSAO, N_MUNDO, MUNDOS_PARADOS, MUNDOS_ANDANDO, SEMENTE))

frevolab 0.1.0 | mundos de 12000 dias, 100 parados e 100 andando | semente 115


## O andar nos tres mercados, e o chao dos mundos parados

O chao nao e enfeite: sem ele nao se sabe quanto do andar medido e andar de verdade e quanto e o
que o sorteio mostra sozinho.

In [2]:
rng_chao = np.random.default_rng(SEMENTE)
mercados, chao_mercado = {}, {}
for arquivo in SERIES:
    retornos = volatilidade.retornos_log(dados.carregar_serie(arquivo)).dropna().to_numpy()
    mercados[arquivo] = lei.andar_da_lei(retornos, JANELA, CELULAS)
    sigma_mercado = float(np.std(retornos))
    vals = {"tv": [], "kl": [], "wasserstein": []}
    for _ in range(MUNDOS_PARADOS):
        parado = rng_chao.normal(0.0, sigma_mercado, retornos.size)
        andar_parado = lei.andar_da_lei(parado, JANELA, CELULAS)
        for chave in vals:
            vals[chave].append(andar_parado[chave])
    chao_mercado[arquivo] = {k: np.asarray(v, dtype=float) for k, v in vals.items()}
chao = {k: np.asarray([np.median(chao_mercado[a][k]) for a in SERIES]) for k in ("tv", "kl", "wasserstein")}

print("%-12s %10s %10s %12s" % ("serie", "TV", "KL", "W1"))
for arquivo in SERIES:
    m = mercados[arquivo]
    print("%-12s %10.4f %10.4f %12.6f" % (arquivo, m["tv"], m["kl"], m["wasserstein"]))
print("%-12s %10.4f %10.4f %12.6f" % ("chao (mediana)", np.median(chao["tv"]),
                                      np.median(chao["kl"]), np.median(chao["wasserstein"])))
print("%-12s %10.4f %10.4f %12.6f" % ("chao (p95)", np.quantile(chao["tv"], 0.95),
                                      np.quantile(chao["kl"], 0.95),
                                      np.quantile(chao["wasserstein"], 0.95)))

serie                TV         KL           W1
sp500.csv       68.2260    53.8127     0.310308
ibov.csv        67.2346    51.8430     0.450445
btc.csv         59.3984    52.9779     0.563722
chao (mediana)    41.8195    22.2807     0.483310
chao (p95)      42.3913    22.6657     0.632026


## O brinquedo declarado: a mesma massa, do miolo para a borda

Duas amostras da mesma lei, e a mesma massa levada do miolo para a borda do suporte declarado. As
duas reguas veem o movimento --- e o veem com tamanhos diferentes, que e o que as amarras do
capitulo deixam.

In [3]:
rng_bring = np.random.default_rng(SEMENTE + 1)
MASSA_BRINQUEDO = 0.05
linhas_bring = []
for _ in range(SORTES_BRINQUEDO):
    base = rng_bring.normal(0.0, 1.0, 4000)
    aresta = float(np.max(np.abs(base)))
    miolo = base.copy()
    borda = base.copy()
    dentro = np.abs(miolo) < 0.25
    escolhidos = np.flatnonzero(dentro)[: int(MASSA_BRINQUEDO * base.size)]
    miolo[escolhidos] += 1.5
    fora = np.abs(borda) > np.quantile(np.abs(borda), 0.90)
    levados = np.flatnonzero(fora)[: int(MASSA_BRINQUEDO * base.size)]
    borda[levados] = np.sign(borda[levados]) * (aresta + 0.1)
    r_miolo = lei.regua(base, miolo, CELULAS)
    r_borda = lei.regua(base, borda, CELULAS)
    linhas_bring.append((r_miolo, r_borda, aresta))

print("%-22s %10s %10s" % ("movimento", "TV", "W1"))
for etiqueta, posicao in (("miolo", 0), ("borda", 1)):
    tv = float(np.median([l[posicao]["tv"] for l in linhas_bring]))
    w1 = float(np.median([l[posicao]["wasserstein"] for l in linhas_bring]))
    print("%-22s %10.4f %10.6f" % (etiqueta, tv, w1))
razao_pinsker = max(l[1]["razao_pinsker"] for l in linhas_bring if np.isfinite(l[1]["razao_pinsker"]))
razao_diametro = max(l[1]["razao_diametro"] for l in linhas_bring if np.isfinite(l[1]["razao_diametro"]))
print("pior razao de Pinsker (lei estimada): %.3f | pior razao do diametro: %.3f"
      % (razao_pinsker, razao_diametro))

movimento                      TV         W1
miolo                      0.0500   0.075000
borda                      0.0497   0.089031
pior razao de Pinsker (lei estimada): 0.290 | pior razao do diametro: 0.283


## A consequencia: a promessa paga em margem

No mundo que anda, a janela que calibra o corte atravessa leis cada vez mais calmas do que a de
hoje. O corte puro --- erguido em dias tranquilos --- e rompido mais do que a conta assinou. A
margem c x W1 entre a lei da janela e a dos dias recentes rebaixa a barra, e a assinatura volta.

In [4]:
rng_andando = np.random.default_rng(SEMENTE + 2)
puras, marginais, margens_pp = [], [], []
for _ in range(MUNDOS_ANDANDO):
    andando = mudanca.andando(N_MUNDO, rng_andando, SIGMA, fator=FATOR_MUNDO)
    serie = pd.Series(andando, index=pd.RangeIndex(andando.size))
    barra = promessa.corte(serie, JANELA, CAUDA)
    margem = promessa.margem_do_andar(serie, JANELA, CAUDA, c=C, recentes=RECENTES)
    com_margem = promessa.corte_com_margem(serie, JANELA, CAUDA, margem=margem)
    depois = slice(N_MUNDO // 2, N_MUNDO)   # a metade em que a lei ja andou
    valido = barra.notna()
    puras.append(float((serie[valido] < barra[valido]).to_numpy()[depois].mean()))
    validos_m = com_margem.notna()
    marginais.append(float((serie[validos_m] < com_margem[validos_m]).to_numpy()[depois].mean()))
    margens_pp.append(100.0 * float(np.nanmean(margem.to_numpy())))

assinada = promessa.entrega_do_corte(JANELA, CAUDA)
print("assinada %.3f%% | corte puro %.3f%% (faixa %.3f a %.3f) | com margem %.3f%% (faixa %.3f a %.3f)"
      % (100 * assinada, 100 * np.median(puras), 100 * np.quantile(puras, 0.05),
         100 * np.quantile(puras, 0.95), 100 * np.median(marginais),
         100 * np.quantile(marginais, 0.05), 100 * np.quantile(marginais, 0.95)))
print("margem media %.4f pp | mundos em que o puro passou da assinada: %d de %d"
      % (float(np.median(margens_pp)), int(sum(1 for p in puras if p > assinada)),
         MUNDOS_ANDANDO))
print("mundos em que o com margem ficou dentro da faixa sorteada: %d de %d"
      % (int(sum(1 for m in marginais if abs(m - assinada) <= 0.02)), MUNDOS_ANDANDO))

assinada 5.138% | corte puro 5.271% (faixa 5.114 a 5.446) | com margem 3.984% (faixa 3.792 a 4.195)
margem media 0.1987 pp | mundos em que o puro passou da assinada: 86 de 100
mundos em que o com margem ficou dentro da faixa sorteada: 100 de 100


In [5]:
# Figura 1: o andar da lei nos tres mercados nas tres reguas, contra o chao.
fig, eixos = plt.subplots(1, 3, figsize=(11.0, 3.6))
for eixo, chave, rotulo in zip(eixos, ("tv", "kl", "wasserstein"),
                               ("variacao total", "divergencia de KL", "Wasserstein-1")):
    valores = [mercados[a][chave] for a in SERIES]
    eixo.bar(range(len(SERIES)), valores, color="#1f4e79")
    eixo.plot(range(len(SERIES)), [float(np.median(chao_mercado[a][chave])) for a in SERIES],
              "o--", color="#c78f2c", lw=1.4, ms=5,
              label="o chao do proprio mercado")
    eixo.set_xticks(range(len(SERIES)))
    eixo.set_xticklabels([a.replace(".csv", "") for a in SERIES], rotation=15, fontsize=8)
    eixo.set_title(rotulo, fontsize=10)
    eixo.legend(fontsize=7)
eixos[0].set_ylabel("andar acumulado")
fig.suptitle("o andar da lei janela a janela, e o que o sorteio mostra sozinho", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E48_regua", 1)
plt.close(fig)
print("figura E48_regua_1 salva")

figura E48_regua_1 salva


In [6]:
# Figura 2: o brinquedo da cauda --- as duas reguas veem o mesmo movimento?
fig, eixo = plt.subplots(figsize=(8.0, 3.8))
tvs = {"miolo": [l[0]["tv"] for l in linhas_bring], "borda": [l[1]["tv"] for l in linhas_bring]}
w1s = {"miolo": [l[0]["wasserstein"] for l in linhas_bring],
       "borda": [l[1]["wasserstein"] for l in linhas_bring]}
posicoes = np.arange(2)
largura = 0.36
eixo.bar(posicoes - largura / 2, [np.median(tvs["miolo"]), np.median(tvs["borda"])],
         largura, color="#1f4e79", label="variacao total")
eixo.bar(posicoes + largura / 2, [np.median(w1s["miolo"]), np.median(w1s["borda"])],
         largura, color="#c78f2c", label="Wasserstein-1")
eixo.set_xticks(posicoes)
eixo.set_xticklabels(["massa no miolo", "massa na borda"])
eixo.set_title("a mesma massa, em dois lugares do suporte", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E48_regua", 2)
plt.close(fig)
print("figura E48_regua_2 salva")

figura E48_regua_2 salva


In [7]:
# Figura 3: a violacao do corte puro e do corte com margem, contra a assinada.
fig, eixo = plt.subplots(figsize=(8.4, 4.0))
eixo.hist(100 * np.array(puras), bins=24, color="#1f4e79", alpha=0.75,
          label="corte puro: mediana %.3f%%" % (100 * np.median(puras)))
eixo.hist(100 * np.array(marginais), bins=24, color="#c78f2c", alpha=0.75,
          label="corte com margem: mediana %.3f%%" % (100 * np.median(marginais)))
eixo.axvline(100 * assinada, color="#b03a2e", ls="--", lw=1.6,
             label="a assinada: %.3f%%" % (100 * assinada))
eixo.set_xlabel("taxa de violacao depois da mudanca (%)")
eixo.set_ylabel("mundos")
eixo.set_title("a promessa no mundo que anda", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E48_regua", 3)
plt.close(fig)
print("figura E48_regua_3 salva")

figura E48_regua_3 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md §9): o criterio de frescor e o hash das
celulas de codigo.

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: nos tres paineis, as barras dos mercados contra o traco do chao; em que paineis a
   barra passa o traco, e se os rankings dos tres mercados coincidem entre os paineis.
2. **Figura 2**: as duas barras de cada posicao, e a diferenca entre elas nas duas posicoes --- a TV
   pagando a borda muito mais do que a W1.
3. **Figura 3**: as duas pilhas de violacao contra a linha da assinada --- a do puro a direita dela,
   a do com margem em cima.

In [8]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
mediana_tv = float(np.median([mercados[a]["tv"] for a in SERIES]))
mediana_kl = float(np.median([mercados[a]["kl"] for a in SERIES]))
mediana_wasserstein = float(np.median([mercados[a]["wasserstein"] for a in SERIES]))

resultado = {
    "regua_mercados": len(SERIES),
    "regua_chao_mundos": MUNDOS_PARADOS,
    "regua_chao_tv": round(float(np.median(chao["tv"])), 4),
    "regua_chao_kl": round(float(np.median(chao["kl"])), 4),
    "regua_chao_wasserstein": round(float(np.median(chao["wasserstein"])), 6),
    "regua_chao_tv_alto": round(float(np.quantile(chao["tv"], 0.95)), 4),
    "regua_chao_wasserstein_alto": round(float(np.quantile(chao["wasserstein"], 0.95)), 6),
    "regua_andar_tv": round(mediana_tv, 4),
    "regua_andar_kl": round(mediana_kl, 4),
    "regua_andar_wasserstein": round(mediana_wasserstein, 6),
    "regua_andar_wasserstein_max": round(max(mercados[a]["wasserstein"] for a in SERIES), 6),
    "regua_brinquedo_massa_pct": round(100 * MASSA_BRINQUEDO, 2),
    "regua_brinquedo_tv_miolo": round(float(np.median(tvs["miolo"])), 4),
    "regua_brinquedo_tv_borda": round(float(np.median(tvs["borda"])), 4),
    "regua_brinquedo_wasserstein_miolo": round(float(np.median(w1s["miolo"])), 6),
    "regua_brinquedo_wasserstein_borda": round(float(np.median(w1s["borda"])), 6),
    "regua_brinquedo_razao_tv": round(float(np.median(tvs["borda"]) / max(np.median(tvs["miolo"]), 1e-12)), 3),
    "regua_brinquedo_razao_wasserstein": round(float(np.median(w1s["borda"]) / max(np.median(w1s["miolo"]), 1e-12)), 3),
    "regua_pinsker_pior": round(razao_pinsker, 3),
    "regua_diametro_pior": round(razao_diametro, 3),
    "regua_mundos_andando": MUNDOS_ANDANDO,
    "regua_assinada_pct": round(100 * assinada, 4),
    "regua_violacao_pura_pct": round(100 * float(np.median(puras)), 4),
    "regua_violacao_pura_alta_pct": round(100 * float(np.quantile(puras, 0.95)), 4),
    "regua_violacao_margem_pct": round(100 * float(np.median(marginais)), 4),
    "regua_margem_media_pp": round(float(np.median(margens_pp)), 4),
    "regua_mundos_puros_acima": int(sum(1 for p in puras if p > assinada)),
    "regua_mundos_margem_dentro": int(sum(1 for m in marginais if abs(m - assinada) <= 0.02)),
}
caminho = Path("lab/resultados/E48_regua.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "regua_mercados": 3,
 "regua_chao_mundos": 100,
 "regua_chao_tv": 41.8195,
 "regua_chao_kl": 22.2807,
 "regua_chao_wasserstein": 0.48331,
 "regua_chao_tv_alto": 42.3913,
 "regua_chao_wasserstein_alto": 0.632026,
 "regua_andar_tv": 67.2346,
 "regua_andar_kl": 52.9779,
 "regua_andar_wasserstein": 0.450445,
 "regua_andar_wasserstein_max": 0.563722,
 "regua_brinquedo_massa_pct": 5.0,
 "regua_brinquedo_tv_miolo": 0.05,
 "regua_brinquedo_tv_borda": 0.0497,
 "regua_brinquedo_wasserstein_miolo": 0.075,
 "regua_brinquedo_wasserstein_borda": 0.089031,
 "regua_brinquedo_razao_tv": 0.995,
 "regua_brinquedo_razao_wasserstein": 1.187,
 "regua_pinsker_pior": 0.29,
 "regua_diametro_pior": 0.283,
 "regua_mundos_andando": 100,
 "regua_assinada_pct": 5.1383,
 "regua_violacao_pura_pct": 5.2714,
 "regua_violacao_pura_alta_pct": 5.4462,
 "regua_violacao_margem_pct": 3.984,
 "regua_margem_media_pp": 0.1987,
 "regua_mundos_puros_acima": 86,
 "regua_mundos_margem_dentro": 100
}
